# 🔄 Recurrent Neural Network (RNN)
**Time Series Forecasting with Keras/TensorFlow**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow version : {tf.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using a **Synthetic Time Series** dataset — 1000 samples, generated from superimposed sine waves with Gaussian noise.

In [ ]:
# Generate dataset if not exists
np.random.seed(42)
t = np.linspace(0, 100, 1000)
y = np.sin(0.1 * t) + 0.5 * np.sin(0.05 * t) + 0.1 * np.random.randn(1000)
df = pd.DataFrame({'time': t, 'value': y})

print(f'Shape   : {df.shape}')
df.head()

## 3. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['time'], df['value'], color='#52a8e0', lw=1.5, label='Value')
ax.set_title('Synthetic Time Series (Sine Waves + Noise)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Data Preprocessing
> RNNs require 3D input: `(batch_size, sequence_length, features)`. We create sequences using a sliding window.

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 20
values = df['value'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(values)

train_size = int(len(scaled_values) * 0.8)
train_data = scaled_values[:train_size]
test_data = scaled_values[train_size - SEQ_LENGTH:]

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape : {X_test.shape}')

## 5. What is an RNN?
> A **Recurrent Neural Network** processes sequential data by maintaining a hidden state that is passed from one time step to the next.

| Architecture | Key Feature | Best For |
|---|---|---|
| **SimpleRNN** | Basic feedback loop | Very short sequences |
| **LSTM** | Cell state + 3 gates (Forget, Input, Output) | Long-term dependencies |
| **GRU** | 2 gates (Reset, Update), no cell state | Faster training, good performance |

## 6. Build RNN Model

In [ ]:
def build_rnn(rnn_type='LSTM', units=64, dropout=0.2, seq_length=20):
    model = keras.Sequential(name=f'{rnn_type}_Model')
    
    if rnn_type == 'LSTM':
        model.add(layers.LSTM(units, return_sequences=False, input_shape=(seq_length, 1)))
    elif rnn_type == 'GRU':
        model.add(layers.GRU(units, return_sequences=False, input_shape=(seq_length, 1)))
    else:
        model.add(layers.SimpleRNN(units, return_sequences=False, input_shape=(seq_length, 1)))
        
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

model = build_rnn(rnn_type='LSTM', units=64, dropout=0.2, seq_length=SEQ_LENGTH)
model.summary()

## 7. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 8. Training History

In [ ]:
hist = pd.DataFrame(history.history)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist['loss'], color='#e05252', lw=2, label='Train')
ax.plot(hist['val_loss'], color='#52a8e0', lw=2, label='Val')
ax.set_title('Training Loss (MSE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Best Val Loss: {hist["val_loss"].min():.4f}')

## 9. Evaluate on Test Set

In [ ]:
y_pred = model.predict(X_test).flatten()
y_pred_inv = scaler.inverse_transform(y_pred.reshape(-1, 1))
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)

print('='*50)
print('          RNN Test Set Results')
print('='*50)
print(f'  MSE : {mse:.4f}')
print(f'  MAE : {mae:.4f}')
print('='*50)

## 10. Actual vs Predicted Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
test_indices = np.arange(len(y_test_inv))
ax.plot(test_indices, y_test_inv, color='#52a8e0', lw=1.5, label='Actual')
ax.plot(test_indices, y_pred_inv, color='#e05252', lw=1.5, linestyle='--', label='Predicted')
ax.set_title('Test Set: Actual vs Predicted', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 11. Multi-Step Forecasting

In [ ]:
forecast_steps = 50
last_seq = scaled_values[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
forecast_scaled = []

for _ in range(forecast_steps):
    pred = model.predict(last_seq, verbose=0)
    forecast_scaled.append(pred[0, 0])
    last_seq = np.append(last_seq[:, 1:, :], np.reshape(pred, (1, 1, 1)), axis=1)

forecast_inv = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1))
last_actual_inv = scaler.inverse_transform(scaled_values[-SEQ_LENGTH:])

fig, ax = plt.subplots(figsize=(14, 5))
hist_steps = 100
ax.plot(np.arange(hist_steps), last_actual_inv[-hist_steps:], color='#52a8e0', lw=2, label='Historical')
ax.plot(np.arange(hist_steps-1, hist_steps-1+forecast_steps), 
        np.vstack((last_actual_inv[-1], forecast_inv)), 
        color='#e05252', lw=2, linestyle='--', label='Forecast')
ax.axvline(hist_steps-1, color='gray', linestyle=':', alpha=0.5, label='Forecast Start')
ax.set_title(f'{forecast_steps}-Step Ahead Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 12. Effect of RNN Type

In [ ]:
rnn_types = ['SimpleRNN', 'GRU', 'LSTM']
results = {}

for rnn_type in rnn_types:
    m = build_rnn(rnn_type=rnn_type, units=64, dropout=0.2, seq_length=SEQ_LENGTH)
    cb = [EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
    m.fit(X_train, y_train, validation_split=0.15, epochs=100, batch_size=32, callbacks=cb, verbose=0)
    
    yp = m.predict(X_test).flatten()
    yp_inv = scaler.inverse_transform(yp.reshape(-1, 1))
    results[rnn_type] = mean_squared_error(y_test_inv, yp_inv)
    print(f'{rnn_type:12s}  Test MSE: {results[rnn_type]:.4f}')

res_df = pd.DataFrame(list(results.items()), columns=['Architecture', 'Test MSE'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(res_df['Architecture'], res_df['Test MSE'], color=['#f4a4a4', '#a4d4f4', '#a4e4b4'], edgecolor='white')
ax.set_title('Architecture Comparison — Test MSE (Lower is Better)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Squared Error')
plt.tight_layout(); plt.show()

## 13. Save Model & Scaler

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
model.save('../models/rnn_model.keras')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved  → models/rnn_model.keras')
print('Scaler saved → models/scaler.pkl')

## 14. Key Takeaways
> - RNNs are designed for **sequential data** (time series, text, audio).
> - **LSTM** and **GRU** solve the vanishing gradient problem of SimpleRNNs.
> - **Sequence length** is a critical hyperparameter: too short loses context, too long increases complexity.
> - Always **scale** time series data before feeding it to neural networks.
> - Multi-step forecasting accumulates error; predictions degrade the further into the future you go.